# Strategy Dimension: AI in the Business description (Item 1)

Extensive margin (`ai_sentence_share`) and intensive margin (`net_tone`, the
mean FinBERT-tone net sentiment P(positive) - P(negative) in [-1, 1]) of
AI-related sentences in Item 1 (Business) of the 10-K. Item 1 is where a firm
frames AI as part of what it does, so it captures strategic positioning.

Depends on the shared caches written by `nlp_features_setup.ipynb`.
Writes `data_clean/indicators/strategy.parquet`.

In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import logging
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.indicators.common.io import (
    cache_path,
    clear_cache,
    load_cached_step,
    save_cached_step,
)
from src.indicators.nlp_features import (
    DIMENSIONS,
    aggregate_dimension,
    score_forward_looking,
    score_sentences,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")
os.environ.setdefault("EDGAR_IDENTITY", "Timo Koba kab.timo3@gmail.com")

FORCE_REFRESH = False
SHARED = "nlp_features"
UNIVERSE = "sp500"
DIM = DIMENSIONS["strategy"]

# Load the shared front-end outputs produced by nlp_features_setup.ipynb.
filings = load_cached_step(SHARED, "filings", UNIVERSE)
sentences_df = load_cached_step(SHARED, "sentences", UNIVERSE)
sentence_totals_df = load_cached_step(SHARED, "sentence_totals", UNIVERSE)
if filings is None or sentences_df is None or sentence_totals_df is None:
    raise RuntimeError(
        "Shared caches not found. Run notebooks/01_ingest_clean/"
        "nlp_features_setup.ipynb first."
    )

item_sentences = sentences_df[sentences_df["item"] == DIM.item].reset_index(drop=True)
n_filings = item_sentences["accession_number"].nunique() if len(item_sentences) else 0
print(f"Dimension: {DIM.name}  |  Item: {DIM.item}  |  tone: {DIM.tone}")
print(f"AI sentences in {DIM.item}: {len(item_sentences)} across {n_filings} filings")

## Score AI sentences

Score the Item's AI sentences with the dimension's tone model
(FinBERT-tone for sentiment dimensions, FinBERT-FLS for the
forward-looking dimension). Per-sentence results are cached on disk by
`sha256(sentence)`, so re-runs are near-free. The scored slice is cached
under this dimension's own namespace.

In [ ]:
scored = None if FORCE_REFRESH else load_cached_step(DIM.name, "scored", UNIVERSE)
if scored is None:
    if len(item_sentences) == 0:
        scored = item_sentences.copy()
    elif DIM.tone == "sentiment":
        scores = score_sentences(item_sentences["sentence"].tolist())
        scored = pd.concat(
            [item_sentences, scores[["pos", "neu", "neg", "label", "confidence"]]],
            axis=1,
        )
    else:  # forward_looking
        scores = score_forward_looking(item_sentences["sentence"].tolist())
        scored = pd.concat(
            [item_sentences, scores[["p_specific", "p_nonspecific", "p_not", "label", "confidence"]]],
            axis=1,
        )
    save_cached_step(scored, DIM.name, "scored", UNIVERSE)
    print(f"Scored {len(scored)} {DIM.item} AI sentences (saved to {cache_path(DIM.name, 'scored', UNIVERSE)})")
else:
    print(f"Loaded {len(scored)} scored sentences from cache ({cache_path(DIM.name, 'scored', UNIVERSE)})")
scored.head()

## Aggregate to firm level and write the indicator parquet

One row per firm. Missing is marked, never conflated with zero: if this
Item did not parse (`item_parsed = 0`), `ai_sentence_share` and
`has_ai_mention` are NaN; a parsed Item with no AI sentences is a genuine 0.
The tone is the mean over all of the Item's AI sentences — NaN only when
there are none (an average over zero sentences is undefined). There is no
minimum-sentence tone threshold: little AI text already shows up in
`ai_sentence_share`, and gating the tone on top would penalise it twice;
`n_ai_sentences` ships with every row so noisy small-sample tones can be
weighted or filtered downstream instead. `parse_complete` still flags firms
where all three Items parsed. Written to
`data_clean/indicators/<UNIVERSE>/<dimension>.parquet`; nothing is dropped
here — missing values are handled at index-composition time.

In [ ]:
indicator = aggregate_dimension(DIM, filings, scored, sentence_totals_df, UNIVERSE)
tone_col = "net_tone" if DIM.tone == "sentiment" else "fls_score"
print(f"{DIM.name}: {len(indicator)} firm rows -> data_clean/indicators/{UNIVERSE}/{DIM.name}.parquet")
print(f"Item parsed (share defined):      {int(indicator['item_parsed'].sum())}/{len(indicator)}")
print(f"Firms with AI mention:            {int((indicator['has_ai_mention'] == 1).sum())}")
print(f"Firms with tone (>= 1 AI sent.):  {int(indicator[tone_col].notna().sum())}")
indicator.head()

## Sanity checks

Coverage split into parsed vs. missing (NaN), the share distribution over
parsed firms, the tone distribution over firms with at least one AI
sentence, and the `n_ai_sentences` distribution that downstream weighting
can lean on. Missing (NaN) and genuine 0 are reported separately.

In [ ]:
tone_col = "net_tone" if DIM.tone == "sentiment" else "fls_score"
parsed = indicator[indicator["item_parsed"] == 1]
n_with_ai = int((parsed["has_ai_mention"] == 1).sum())

print("=== Coverage ===")
print(f"  total firms:                 {len(indicator)}")
print(f"  item parsed (share defined): {len(parsed)}   (missing/NaN: {len(indicator) - len(parsed)})")
print(f"  parse_complete (all Items):  {int(indicator['parse_complete'].sum())}")
if len(parsed):
    print(f"  with AI mention:             {n_with_ai}  ({n_with_ai / len(parsed):.2%} of parsed)")
    print(f"  ai_sentence_share mean:      {parsed['ai_sentence_share'].mean():.4f}  sd: {parsed['ai_sentence_share'].std():.4f}")

sub = indicator.loc[indicator[tone_col].notna(), tone_col]
print(f"\n=== Tone ({tone_col}) ===")
print(f"  firms with tone: {len(sub)}  (= parsed firms with >= 1 AI sentence; zero AI sentences -> NaN, not 0)")
if len(sub):
    print(f"  mean: {sub.mean():+.4f}  sd: {sub.std():.4f}")

_n_ai = parsed.loc[parsed["n_ai_sentences"] > 0, "n_ai_sentences"]
print("\n=== n_ai_sentences distribution (AI-mentioning firms) ===")
print(_n_ai.describe().to_string() if len(_n_ai) else "(none)")